In [316]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [317]:
df_articles = pd.read_csv('data/articles.csv')

In [318]:
numeric_cols = df_articles.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = df_articles.select_dtypes(include=['object']).columns.tolist()
print("Numeric columns:", {len(numeric_cols): numeric_cols})
print("Categorical columns:", {len(categorical_cols): categorical_cols})

Numeric columns: {11: ['article_id', 'product_code', 'product_type_no', 'graphical_appearance_no', 'colour_group_code', 'perceived_colour_value_id', 'perceived_colour_master_id', 'department_no', 'index_group_no', 'section_no', 'garment_group_no']}
Categorical columns: {14: ['prod_name', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'perceived_colour_master_name', 'department_name', 'index_code', 'index_name', 'index_group_name', 'section_name', 'garment_group_name', 'detail_desc']}


In [319]:
# Check the number of null values per column (initial status)
missing_counts = df_articles.isnull().sum().sort_values(ascending=False)
print("\nTop columns with most missing values (count):")
print(missing_counts.head(10))


Top columns with most missing values (count):
detail_desc                     416
perceived_colour_master_name      0
garment_group_name                0
garment_group_no                  0
section_name                      0
section_no                        0
index_group_name                  0
index_group_no                    0
index_name                        0
index_code                        0
dtype: int64


In [320]:
# Kết hợp các cột mô tả thành một cột duy nhất
import pandas as pd
import re

# Columns to combine
cols = [
    'prod_name',
    'product_type_name',
    'product_group_name',
    'graphical_appearance_name',
    'colour_group_name',
    'perceived_colour_value_name',
    'perceived_colour_master_name',
    'index_name',
    'index_group_name',
    'section_name'
]

# Fill NA
df_articles[cols] = df_articles[cols].fillna('')

# -------------------------
# Function to clean + deduplicate
# -------------------------
def build_description(row):
    # Combine all text
    text = ' '.join([str(row[col]) for col in cols])
    
    # Lowercase
    text = text.lower()
    
    # Remove special characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # Split into words
    words = text.split()
    
    # Remove duplicates while keeping order
    seen = set()
    unique_words = []
    
    for w in words:
        if w not in seen:
            seen.add(w)
            unique_words.append(w)
    
    return ' '.join(unique_words)

# Apply
df_articles['clothes_description'] = df_articles.apply(build_description, axis=1)

In [321]:
synonyms = {
    'baby/children': ['baby', 'kids', 'children'],
    'ladieswear': ['women', 'female'],
    'menswear': ['men', 'male']
}

def add_synonyms(text):
    words = text.split()
    expanded = set(words)
    
    for w in words:
        if w in synonyms:
            expanded.update(synonyms[w])
    
    return ' '.join(expanded)

df_articles['clothes_description'] = df_articles['clothes_description'].apply(add_synonyms)

stopwords = {'the', 'and', 'for', 'with'}

df_articles['clothes_description'] = df_articles['clothes_description'].apply(
    lambda x: ' '.join([w for w in x.split() if w not in stopwords])
)

In [322]:
df_customer = pd.read_csv('data/customers.csv')

In [323]:
def missing_data(df_customer):
    """
    missing_data Computes Percentage of Missing Values for the dataframe

    Parameters
    ----------
    df_customer : dataframe
        Dataframe for which the missing value percentages need to be computed.

    Returns
    -------
    missing_df: dataframe
        Dataframe with missing value percentages.
    """
    total = df_customer.isnull().sum().sort_values(ascending=False)
    percent = (df_customer.isnull().sum() / df_customer.isnull().count() * 100).sort_values(
        ascending=False
    )
    missing_df = pd.concat([total, percent], axis=1, keys=["Total", "Percent"])
    return missing_df

missing_data(df_customer)

,Total,Percent
Active,907576,66.150819
FN,895050,65.237831
fashion_news_frequency,16011,1.167000
age,15861,1.156066
club_member_status,6062,0.441843
customer_id,0,0.000000
postal_code,0,0.000000


In [324]:
# Handle nulls in customers
df_customer['FN'] = df_customer['FN'].fillna(0)
df_customer['Active'] = df_customer['Active'].fillna(0)
df_customer['FN'] = df_customer['FN'].astype(int)
df_customer['Active'] = df_customer['Active'].astype(int) # convert to int for better handling 
df_customer['fashion_news_frequency'] = df_customer['fashion_news_frequency'].fillna('NONE')
df_customer['club_member_status'] = df_customer['club_member_status'].fillna('Non-Member')
df_customer['age'] = df_customer.groupby('club_member_status')['age'].transform(lambda x: x.fillna(x.median())) # Fill age with median age of the respective club_member_status group


df_customer['age_group'] = pd.cut(df_customer['age'], bins=[0, 20, 30, 40, 50, 60, 100], labels=['&lt;20', '20-30', '30-40', '40-50', '50-60', '60+'])      

In [325]:
# User features - keep numeric binary flags and one-hot encode only categorical variables
user_features = pd.get_dummies(
    df_customer[['club_member_status', 'fashion_news_frequency', 'age_group']])

# Retain FN and Active as numeric features
df_customer = pd.concat(
    [df_customer[['customer_id', 'FN', 'Active']], user_features],
    axis=1
)

In [326]:
df_transactions = pd.read_csv('data/transactions.csv')

df_transactions['t_dat'] = pd.to_datetime(df_transactions['t_dat'])

In [327]:
missing_data(df_transactions)

,Total,Percent
t_dat,0,0.0
customer_id,0,0.0
article_id,0,0.0
price,0,0.0
sales_channel_id,0,0.0


In [331]:
# Transaction features (using 5-week window only)

# Purchase frequency per user
user_purchase_count = df_transactions_recent.groupby('customer_id').size().reset_index(name='purchase_count')

# Item popularity
item_popularity = df_transactions_recent.groupby('article_id').size().reset_index(name='popularity')

# Average price per user
user_avg_price = df_transactions_recent.groupby('customer_id')['price'].mean().reset_index(name='avg_price')

# Last purchase date per user
user_last_purchase = df_transactions_recent.groupby('customer_id')['t_dat'].max().reset_index(name='last_purchase')

# Recency (days since last purchase in the 5-week window)
max_date = df_transactions_recent['t_dat'].max()
user_last_purchase['recency'] = (max_date - user_last_purchase['last_purchase']).dt.days

# Merge user features
df_customer = df_customer.merge(user_purchase_count, on='customer_id', how='left').fillna(0)
df_customer = df_customer.merge(user_avg_price, on='customer_id', how='left').fillna(0)
df_customer = df_customer.merge(user_last_purchase[['customer_id', 'recency']], on='customer_id', how='left').fillna(999)

# For items, convert article_id to int for merge compatibility
item_popularity['article_id'] = item_popularity['article_id'].astype(int)
df_articles = df_articles.merge(item_popularity, on='article_id', how='left').fillna(0)

print('User features merged. Customer shape:', df_customer.shape)
print('Item features merged. Article shape:', df_articles.shape)

User features merged. Customer shape: (1371980, 25)
Item features merged. Article shape: (105542, 28)
